# Классификация ботов
Финальная версия решения с временной валидацией.

Подключаю основные библиотеки.

In [2]:
import numpy as np
import pandas as pd

In [3]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [4]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


## Просмотр данных

In [5]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print(events.user_agent.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

user_agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 YaBrowser/23.11.0.0 Safari/537.36    3737
Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:119.0) Gecko/20100101 Firefox/119.0                                                       3670
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 YaBrowser/23.9.0.0 Safari/537.36     3

## Окно наблюдения
Оставляю только события внутри окна cookie.

In [6]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


Проверяю, что train и test события собрались.

In [7]:
print("train\n")
print(ev_tr.head())
print("\n\n")
print("test\n")
print(ev_te.head())

train

             cookie_id            event_ts  eid           event_name platform  \
0  ck_d274382b19488771 2026-04-13 12:02:56  200            item_view      WEB   
1  ck_fbbed2ff14944ce9 2026-04-08 06:12:14  100  search_results_view      web   
2  ck_56cc15c7c634cb9f 2026-04-12 17:16:05  100  search_results_view      WEB   
4  ck_f80d85f038cdacee 2026-04-11 16:08:12  200            item_view  desktop   
5  ck_e4f046f721d10f81 2026-04-10 16:50:05  200            item_view      IOS   

                                          user_agent    item_id  \
0  Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...  1048743.0   
1  Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...        NaN   
2  Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...        NaN   
4  Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...  1000845.0   
5  Mozilla/5.0 (iPhone; CPU iPhone OS 15_2 like M...  1055188.0   

       item_category    item_location seller_type    search_query  \
0  kvartiry_prodazha            ki

## Простые признаки

In [8]:
def basic_features(ev, meta):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
    })
    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)

Xtr = basic_features(ev_tr, train)
Xte = basic_features(ev_te, test)
ytr = train.target.values
Xtr.head()

,cookie_id,n_events,item_nunique
0,ck_54a059eb7d3ea68b,7,4
1,ck_7e4de46eeab82974,41,19
2,ck_9320229ef6304522,36,19
3,ck_30ccd25bc1714ed9,21,10
4,ck_a77c5f05948cdeef,29,16


In [9]:
from sklearn.ensemble import RandomForestClassifier
from metric import precision_at_recall   # официальная реализация, ей же считает автопроверка

is_valid = train.window_start_ts.ge('2026-04-17').values
cols = ['n_events', 'item_nunique']

model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
model.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid])
p_va = model.predict_proba(Xtr.loc[is_valid, cols])[:, 1]

print('P@R0.7 на валидации:', round(precision_at_recall(ytr[is_valid], p_va), 4))
print('доля ботов (это уровень константы):', round(ytr[is_valid].mean(), 4))

P@R0.7 на валидации: 0.1025
доля ботов (это уровень константы): 0.082


In [10]:
model.fit(Xtr[cols], ytr)
sub = pd.DataFrame({
    'cookie_id': Xte.cookie_id,
    'score': model.predict_proba(Xte[cols])[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.052951
1,ck_a76ee3b3e3e522fd,0.058161
2,ck_94c9a4d382689e82,0.037449
3,ck_8eaf9509ad9462a0,0.058161
4,ck_9a88a5a989cb5bc6,0.034818


## 1. Подготовка данных
Привожу данные к удобному виду.

In [11]:
events['platform'] = events['platform'].str.lower()
events['user_agent'] = events['user_agent'].str.lower() # делаем строчные буквы везде

events = events.drop(columns=['eid']) # этот столбец дублирует event_name

print(events.shape)

events = events.drop_duplicates() # удаляем строчки лишние

print(events.shape)

meta = pd.concat([
    train[['cookie_id', 'window_start_ts', 'window_end_ts']],
    test[['cookie_id', 'window_start_ts', 'window_end_ts']]
], ignore_index=True)
events = events.merge(meta, on='cookie_id', how='inner')
events = events[
    (events['event_ts'] >= events['window_start_ts']) & 
    (events['event_ts'] < events['window_end_ts'])
]
events = events.drop(columns=['window_start_ts', 'window_end_ts']).reset_index(drop=True) # отсеиваем строчки все окна 

print(events.shape) # ура минус 45к строк

(328905, 13)
(324037, 13)
(283830, 13)


In [12]:
print(events['event_ts'].dtypes)

datetime64[ns]


In [13]:
print(events['user_agent'].unique())

['mozilla/5.0 (macintosh; intel mac os x 10_15_7) applewebkit/537.36 (khtml, like gecko) chrome/119.0.0.0 safari/537.36'
 'mozilla/5.0 (windows nt 10.0; win64; x64; rv:113.0) gecko/20100101 firefox/113.0'
 'mozilla/5.0 (windows nt 10.0; win64; x64) applewebkit/537.36 (khtml, like gecko) chrome/113.0.0.0 yabrowser/23.5.0.0 safari/537.36'
 'mozilla/5.0 (macintosh; intel mac os x 10_15_7) applewebkit/537.36 (khtml, like gecko) chrome/125.0.0.0 safari/537.36'
 'mozilla/5.0 (macintosh; intel mac os x 10_15_7) applewebkit/537.36 (khtml, like gecko) chrome/115.0.0.0 safari/537.36'
 'avito/122.0 (android 11; m2101k6g) okhttp/4.11.0'
 'mozilla/5.0 (windows nt 10.0; win64; x64) applewebkit/537.36 (khtml, like gecko) chrome/123.0.0.0 yabrowser/23.3.0.0 safari/537.36'
 'mozilla/5.0 (iphone; cpu iphone os 15_2 like mac os x) applewebkit/605.1.15 (khtml, like gecko) version/15.0 mobile/15e148 safari/604.1'
 'mozilla/5.0 (windows nt 10.0; win64; x64) applewebkit/537.36 (khtml, like gecko) chrome/114.

извлекаем признаки из User_Agent.

In [14]:
import numpy as np
def process_user_agent_features(df, ua_column='user_agent'):
    ua = df[ua_column].astype(str).str.lower()
    
    # 1. Тип клиента (Бинарные флаги)
    df['ua_is_avito_app'] = ua.str.contains('avito/').astype(int)
    df['ua_is_script_bot'] = ua.str.contains('python|urllib3|requests|scrapy|curl|go-http-client|node-fetch').astype(int)
    df['ua_is_headless'] = ua.str.contains('headlesschrome').astype(int)
    df['ua_is_browser'] = (ua.str.startswith('mozilla/5.0') & (df['ua_is_headless'] == 0)).astype(int)
    
    # 2. Выделение ОС
    conditions_os = [
        ua.str.contains('android'),
        ua.str.contains('iphone|ios'),
        ua.str.contains('windows nt'),
        ua.str.contains('macintosh|mac os x'),
        ua.str.contains('linux|x11')
    ]
    choices_os = ['android', 'ios', 'windows', 'macos', 'linux']
    df['ua_os'] = pd.Series(np.select(conditions_os, choices_os, default='unknown'))
    
    # 3. Выделение семейства браузера / библиотеки
    conditions_browser = [
        ua.str.contains('avito/'),
        ua.str.contains('yabrowser'),
        ua.str.contains('headlesschrome'),
        ua.str.contains('chrome'),
        ua.str.contains('firefox'),
        ua.str.contains('safari') & ~ua.str.contains('chrome'),
        df['ua_is_script_bot'] == 1
    ]
    choices_browser = ['avito_app', 'yandex', 'headless_chrome', 'chrome', 'firefox', 'safari', 'script']
    df['ua_browser'] = pd.Series(np.select(conditions_browser, choices_browser, default='other'))
    
    # 4. Версия приложения Avito (если есть)
    df['ua_avito_ver'] = ua.str.extract(r'avito/(\d+)\.').fillna(-1).astype(float)
    
    # 5. Длина строки User-Agent (у скриптов она короткая, у браузеров длинная)
    df['ua_length'] = df[ua_column].astype(str).str.len()
    
    return df
events = process_user_agent_features(events)

## Категории и города

In [15]:
print(events['item_location'].value_counts())
print(events['item_category'].value_counts())

item_location
sankt-peterburg     34576
novosibirsk         33637
moskva              33448
ekaterinburg        33184
ryazan               4067
izhevsk              4047
barnaul              4004
astrahan             3994
vladivostok          3952
krasnodar            3911
ufa                  3910
tyumen               3877
ulyanovsk            3869
rostov-na-donu       3867
kirov                3858
kazan                3835
omsk                 3833
volgograd            3812
yaroslavl            3793
kemerovo             3792
saratov              3786
tomsk                3772
voronezh             3753
krasnoyarsk          3708
nizhniy_novgorod     3675
ivanovo              3674
tolyatti             3656
cheboksary           3655
perm                 3650
chelyabinsk          3636
orenburg             3622
tula                 3615
lipetsk              3613
habarovsk            3590
penza                3550
bryansk              3541
makhachkala          3526
irkutsk              350

In [16]:
print(events['search_query'].value_counts())
print(events['search_query'].unique())


search_query
renault logan             826
toyota camry              815
кофемашина                813
kia rio 2019              812
посудомоечная машина      809
                         ... 
аккумулятор               670
тормозные колодки         664
samsung s23               662
снять квартиру у метро    659
macbook air m1            652
Name: count, Length: 120, dtype: int64
[nan 'iphone 13 128' 'пылесос dyson' 'костюм мужской' 'лыжи' 'лада гранта'
 'репетитор математика' 'работа грузчиком' 'клетка для попугая'
 'щенки в добрые руки' 'кухонный гарнитур' 'манеж' 'апартаменты аренда'
 'коляска 2в1' 'купить 1к квартиру' 'стиральная машина' 'новостройка'
 'велосипед детский' 'аренда комнаты' 'котята' 'lenovo thinkpad'
 'конструктор lego' 'хомяк' 'kia rio 2019' 'ноутбук игровой' 'шкаф купе'
 'смена 12 часов' 'dell xps' 'тормозные колодки' 'апартаменты купить'
 'телефон бу' 'macbook pro 14' 'диван угловой' 'корм для собак' 'электрик'
 'кресло компьютерное' 'снять квартиру у метро' 'toyot

In [17]:
print(events.head())
print(events.info())

             cookie_id            event_ts           event_name platform  \
0  ck_5efbea1befdefe1b 2026-04-26 09:11:24            item_view  desktop   
1  ck_c4ca1434f3778f1d 2026-04-20 14:04:31  search_results_view      web   
2  ck_d274382b19488771 2026-04-13 12:02:56            item_view      web   
3  ck_fbbed2ff14944ce9 2026-04-08 06:12:14  search_results_view      web   
4  ck_56cc15c7c634cb9f 2026-04-12 17:16:05  search_results_view      web   

                                          user_agent    item_id  \
0  mozilla/5.0 (macintosh; intel mac os x 10_15_7...  1027450.0   
1  mozilla/5.0 (windows nt 10.0; win64; x64; rv:1...        NaN   
2  mozilla/5.0 (windows nt 10.0; win64; x64) appl...  1048743.0   
3  mozilla/5.0 (macintosh; intel mac os x 10_15_7...        NaN   
4  mozilla/5.0 (macintosh; intel mac os x 10_15_7...        NaN   

       item_category    item_location seller_type    search_query  ...  \
0        elektronika      kaliningrad         pro             NaN 

временные признаки

In [18]:
import pandas as pd
import numpy as np

def extract_ts_features(df: pd.DataFrame, cookie_col: str = 'cookie_id', ts_col: str = 'event_ts') -> pd.DataFrame:
    df_ts = df[[cookie_col, ts_col]].copy()
    
    # Извлекаем временные метки
    df_ts['hour'] = df_ts[ts_col].dt.hour
    df_ts['day'] = df_ts[ts_col].dt.date
    df_ts['is_night'] = df_ts['hour'].isin([0, 1, 2, 3, 4, 5]).astype(int)
    
    features = df_ts.groupby(cookie_col).agg(
        # Основной день сессии
        event_day=('day', lambda x: x.mode()[0] if not x.empty else np.nan),
        
        # Общая временная активность
        session_duration_sec=(ts_col, lambda x: (x.max() - x.min()).total_seconds()),
        events_count=(ts_col, 'count'),
        
        # Динамика времени
        min_hour=('hour', 'min'),
        max_hour=('hour', 'max'),
        hour_std=('hour', 'std'),
        night_events_share=('is_night', 'mean')
    ).reset_index()
    
    # Заполняем std для кук с 1 событием
    features['hour_std'] = features['hour_std'].fillna(0)
    
    return features

In [19]:
def extract_event_name_features(df: pd.DataFrame, cookie_col: str = 'cookie_id', event_col: str = 'event_name') -> pd.DataFrame:
    # Кросс-табуляция и нормализация по строке (доли)
    event_shares = pd.crosstab(df[cookie_col], df[event_col], normalize='index')
    
    # Переименовываем колонки в event_share_<name>
    event_shares.columns = [f'event_share_{col}' for col in event_shares.columns]
    
    return event_shares.reset_index()

признаки платформ

In [20]:
def extract_platform_features(df: pd.DataFrame, cookie_col: str = 'cookie_id', platform_col: str = 'platform') -> pd.DataFrame:
    features = df.groupby(cookie_col).agg(
        main_platform=(platform_col, lambda x: x.mode()[0] if not x.empty else 'unknown'),
        unique_platforms_count=(platform_col, 'nunique')
    ).reset_index()
    
    return features

признаки User_Agent

In [21]:
def extract_ua_features(df: pd.DataFrame, cookie_col: str = 'cookie_id') -> pd.DataFrame:
    features = df.groupby(cookie_col).agg(
        # Флаги автоматизации и типов
        has_script_bot=('ua_is_script_bot', 'max'),
        has_headless=('ua_is_headless', 'max'),
        has_avito_app=('ua_is_avito_app', 'max'),
        has_browser=('ua_is_browser', 'max'),
        
        # Разнообразие
        unique_ua_count=('user_agent', 'nunique'),
        unique_os_count=('ua_os', 'nunique'),
        unique_browser_count=('ua_browser', 'nunique'),
        
        # Статистики
        mean_ua_length=('ua_length', 'mean'),
        max_avito_ver=('ua_avito_ver', 'max'),
        
        # Преобладающие категории
        main_os=('ua_os', lambda x: x.mode()[0] if not x.empty else 'unknown'),
        main_browser=('ua_browser', lambda x: x.mode()[0] if not x.empty else 'unknown')
    ).reset_index()
    
    return features

In [22]:
TOP_MEGAPOLISES = {'sankt-peterburg', 'novosibirsk', 'moskva', 'ekaterinburg'}

import pandas as pd
import numpy as np

def extract_item_context_features(df: pd.DataFrame, cookie_col: str = 'cookie_id') -> pd.DataFrame:
    df_item = df.copy()
    
    # 1. Векторизованный флаг не-мегаполиса
    df_item['is_top_megapolis'] = df_item['item_location'].isin(TOP_MEGAPOLISES).astype(int)
    df_item['is_non_megapolis'] = (df_item['is_top_megapolis'] == 0).astype(int)
    
    # Вспомогательная функция для безопасного взятия моды
    def safe_mode(s: pd.Series) -> str:
        m = s.mode()
        return m.iloc[0] if not m.empty else 'unknown'

    # 2. Агрегация
    features = df_item.groupby(cookie_col).agg(
        # item_id
        unique_items_count=('item_id', 'nunique'),
        total_item_views=('item_id', 'count'),
        
        # item_location
        unique_locations_count=('item_location', 'nunique'),
        has_top_megapolis=('is_top_megapolis', 'max'),
        non_megapolis_views_count=('is_non_megapolis', 'sum'),
        
        # item_category
        unique_categories_count=('item_category', 'nunique'),
        main_category=('item_category', safe_mode)
    ).reset_index()

    # 3. Доп. метрики
    features['events_per_item'] = np.where(
        features['unique_items_count'] > 0,
        features['total_item_views'] / features['unique_items_count'],
        0.0
    )
    
    return features

признаки поиска и продавцов

In [23]:
def extract_search_and_seller_features(df: pd.DataFrame, cookie_col: str = 'cookie_id') -> pd.DataFrame:
    df_s = df[[cookie_col, 'search_query', 'search_page', 'seller_type']].copy()
    
    # Векторизованные флаги (работают без вызова лямбда-функций)
    df_s['has_search'] = df_s['search_query'].notna().astype(int)
    df_s['is_pro_seller'] = (df_s['seller_type'] == 'pro').astype(int)
    df_s['is_private_seller'] = (df_s['seller_type'] == 'private').astype(int)
    
    features = df_s.groupby(cookie_col).agg(
        # Поиск
        search_count=('has_search', 'sum'),
        unique_search_queries_count=('search_query', 'nunique'),
        max_search_page=('search_page', 'max'),
        
        # Продавцы
        pro_seller_views=('is_pro_seller', 'sum'),
        private_seller_views=('is_private_seller', 'sum')
    ).reset_index()
    
    # Разнообразие поиска
    features['query_diversity'] = np.where(
        features['search_count'] > 0,
        features['unique_search_queries_count'] / features['search_count'],
        0.0
    )
    
    # Заполнение NaN для поисковых страниц
    features['max_search_page'] = features['max_search_page'].fillna(0)
    
    return features

признаки указателя мыши

In [24]:
def extract_pointer_features(df: pd.DataFrame, cookie_col: str = 'cookie_id') -> pd.DataFrame:
    df_p = df.copy()
    df_p['has_pointer'] = (df_p['pointer_x'].notna() & df_p['pointer_y'].notna()).astype(int)
    
    features = df_p.groupby(cookie_col).agg(
        # Наличие указателя
        pointer_events_count=('has_pointer', 'sum'),
        pointer_events_share=('has_pointer', 'mean'),
        
        # Разнообразие точек клика
        unique_x_count=('pointer_x', 'nunique'),
        unique_y_count=('pointer_y', 'nunique'),
        
        # Дисперсия/Стандартное отклонение координат (0.0 = кликает строго в одну точку)
        std_pointer_x=('pointer_x', 'std'),
        std_pointer_y=('pointer_y', 'std'),
        
        # Средние координаты
        mean_pointer_x=('pointer_x', 'mean'),
        mean_pointer_y=('pointer_y', 'mean')
    ).reset_index()
    
    # Заполнение пропусков у сессий без кликов/с 1 кликом
    features['std_pointer_x'] = features['std_pointer_x'].fillna(-1)
    features['std_pointer_y'] = features['std_pointer_y'].fillna(-1)
    features['mean_pointer_x'] = features['mean_pointer_x'].fillna(-1)
    features['mean_pointer_y'] = features['mean_pointer_y'].fillna(-1)
    
    return features

собираем один датасет на cookie

In [25]:
def build_cookie_dataset(events_df: pd.DataFrame) -> pd.DataFrame:
    """Единая сборка датасета: 1 строка = 1 cookie_id."""
    
    print("1/7 Извлечение временных признаков...")
    df_ts = extract_ts_features(events_df)
    
    print("2/7 Извлечение долей событий...")
    df_events = extract_event_name_features(events_df)
    
    print("3/7 Извлечение платформ...")
    df_platform = extract_platform_features(events_df)
    
    print("4/7 Извлечение User-Agent признаков...")
    df_ua = extract_ua_features(events_df)
    
    print("5/7 Извлечение признаков объявлений и категорий...")
    df_item = extract_item_context_features(events_df)
    
    print("6/7 Извлечение признаков поиска и продавцов...")
    df_search = extract_search_and_seller_features(events_df)
    
    print("7/7 Извлечение признаков указателя мыши...")
    df_pointer = extract_pointer_features(events_df)
    
    # Последовательное объединение по cookie_id
    cookie_df = df_ts.merge(df_events, on='cookie_id', how='left')
    cookie_df = cookie_df.merge(df_platform, on='cookie_id', how='left')
    cookie_df = cookie_df.merge(df_ua, on='cookie_id', how='left')
    cookie_df = cookie_df.merge(df_item, on='cookie_id', how='left')
    cookie_df = cookie_df.merge(df_search, on='cookie_id', how='left')
    cookie_df = cookie_df.merge(df_pointer, on='cookie_id', how='left')
    
    print("Завершено! Итоговый размер датасета:", cookie_df.shape)
    return cookie_df

# Запуск сборки:
cookie_dataset = build_cookie_dataset(events)

1/7 Извлечение временных признаков...
2/7 Извлечение долей событий...
3/7 Извлечение платформ...
4/7 Извлечение User-Agent признаков...
5/7 Извлечение признаков объявлений и категорий...
6/7 Извлечение признаков поиска и продавцов...
7/7 Извлечение признаков указателя мыши...
Завершено! Итоговый размер датасета: (16000, 52)


### собранный датасет:

In [26]:
cookie_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 52 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   cookie_id                         16000 non-null  object 
 1   event_day                         16000 non-null  object 
 2   session_duration_sec              16000 non-null  float64
 3   events_count                      16000 non-null  int64  
 4   min_hour                          16000 non-null  int32  
 5   max_hour                          16000 non-null  int32  
 6   hour_std                          16000 non-null  float64
 7   night_events_share                16000 non-null  float64
 8   event_share_contact_chat_open     16000 non-null  float64
 9   event_share_contact_message_sent  16000 non-null  float64
 10  event_share_contact_phone_show    16000 non-null  float64
 11  event_share_favorite_add          16000 non-null  float64
 12  even

## 1.9. Расширенные признаки
Добавляем новые поведенческие признаки

In [27]:
# -----------------------------
# 1) BASE: текущий рабочий набор
# -----------------------------
BASE_COOKIE_FEATURES_RAW = [c for c in cookie_dataset.columns if c != 'cookie_id']

# Возраст cookie к началу окна — строго доступен до наблюдаемого окна.
meta_age = pd.concat([
    train[['cookie_id', 'window_start_ts', 'cookie_created_at']],
    test[['cookie_id', 'window_start_ts', 'cookie_created_at']],
], ignore_index=True).drop_duplicates('cookie_id')
meta_age['account_age_days'] = (
    meta_age['window_start_ts'] - meta_age['cookie_created_at']
).dt.total_seconds() / 86400.0
cookie_dataset = cookie_dataset.merge(
    meta_age[['cookie_id', 'account_age_days']], on='cookie_id', how='left', validate='one_to_one'
)

# Базовые inter-event gaps: сохраняем именно тот сигнал, который уже дал заметный прирост.
ev_sorted = events.sort_values(['cookie_id', 'event_ts'])
gaps = ev_sorted.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
gap_features = (
    pd.DataFrame({'cookie_id': ev_sorted['cookie_id'], 'gap_sec': gaps})
    .dropna()
    .groupby('cookie_id')['gap_sec']
    .agg(gap_mean='mean', gap_std='std', gap_median='median', gap_min='min')
    .reset_index()
)
gap_features['gap_cv'] = (
    gap_features['gap_std'] / gap_features['gap_mean'].replace(0, np.nan)
)
cookie_dataset = cookie_dataset.merge(gap_features, on='cookie_id', how='left', validate='one_to_one')

# Это наш контрольный BASE-набор до новых поведенческих признаков.
BASE_COOKIE_FEATURES = cookie_dataset.columns.tolist()


Определяю вспомогательные функции для новых признаков.

In [28]:

# -----------------------------
# 2) EXTENDED: новые поведенческие признаки
# -----------------------------
def _max_events_in_window(sec_values, window_sec: int) -> int:
    x = np.asarray(sec_values, dtype=np.float64)
    if x.size == 0:
        return 0
    x = np.sort(x)
    right = np.searchsorted(x, x + window_sec, side='right')
    return int(np.max(right - np.arange(x.size)))


def _distribution_metrics(work: pd.DataFrame, column: str, prefix: str) -> pd.DataFrame:
    sub = work.loc[work[column].notna(), ['cookie_id', column]]
    if sub.empty:
        return pd.DataFrame(columns=[
            'cookie_id', f'{prefix}_count', f'{prefix}_unique',
            f'{prefix}_unique_ratio', f'{prefix}_top_share',
            f'{prefix}_repeat_ratio', f'{prefix}_entropy'
        ])
    cnt = sub.groupby(['cookie_id', column], dropna=False).size().rename('n').reset_index()
    agg = cnt.groupby('cookie_id')['n'].agg(total='sum', unique='size', top='max')
    p = cnt['n'] / cnt.groupby('cookie_id')['n'].transform('sum')
    ent = (-p * np.log2(np.clip(p, 1e-12, None))).groupby(cnt['cookie_id']).sum()
    return pd.DataFrame({
        'cookie_id': agg.index,
        f'{prefix}_count': agg['total'].to_numpy(),
        f'{prefix}_unique': agg['unique'].to_numpy(),
        f'{prefix}_unique_ratio': (agg['unique'] / agg['total']).to_numpy(),
        f'{prefix}_top_share': (agg['top'] / agg['total']).to_numpy(),
        f'{prefix}_repeat_ratio': (1 - agg['unique'] / agg['total']).to_numpy(),
        f'{prefix}_entropy': ent.reindex(agg.index).fillna(0).to_numpy(),
    })


Считаю расширенные поведенческие признаки.

In [29]:


def build_extended_behavior_features(events_df: pd.DataFrame, meta_df: pd.DataFrame) -> pd.DataFrame:
    # events_df уже отфильтрован по window_start_ts <= event_ts < window_end_ts.
    meta_small = meta_df[['cookie_id', 'window_start_ts', 'window_end_ts']].drop_duplicates('cookie_id')
    w = events_df.merge(meta_small, on='cookie_id', how='left', validate='many_to_one')
    w = w.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)
    w['rel_sec'] = (w['event_ts'] - w['window_start_ts']).dt.total_seconds()
    w['hour'] = w['event_ts'].dt.hour
    w['gap_sec'] = w.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()

    first_mask = w.groupby('cookie_id').cumcount().eq(0)
    w['event_change'] = w['event_name'].ne(w.groupby('cookie_id')['event_name'].shift()).astype(float)
    w.loc[first_mask, 'event_change'] = np.nan
    for col in ['platform', 'user_agent', 'ua_browser', 'ua_os']:
        if col in w.columns:
            sw = w[col].ne(w.groupby('cookie_id')[col].shift()).astype(float)
            sw.loc[first_mask] = np.nan
            w[f'{col}_switch'] = sw

    out = pd.DataFrame({'cookie_id': w['cookie_id'].unique()})

    # Частота и плотность.
    timing = w.groupby('cookie_id')['event_ts'].agg(
        first_event='min', last_event='max', events_ext='size'
    ).reset_index()
    timing['active_span_sec_ext'] = (
        timing['last_event'] - timing['first_event']
    ).dt.total_seconds().fillna(0)
    timing['active_span_ratio'] = timing['active_span_sec_ext'] / 86400.0
    timing['events_per_hour_span'] = timing['events_ext'] / (
        timing['active_span_sec_ext'] / 3600.0 + 1/60
    )
    out = out.merge(
        timing.drop(columns=['first_event', 'last_event']), on='cookie_id', how='left'
    )

    for minutes in [1, 5, 15, 60]:
        bins = (w['rel_sec'] // (minutes * 60)).astype('int64')
        active_bins = (
            pd.DataFrame({'cookie_id': w['cookie_id'], '_bin': bins})
            .groupby('cookie_id')['_bin'].nunique()
            .rename(f'active_bins_{minutes}m')
        )
        out = out.merge(active_bins, on='cookie_id', how='left')
        out[f'events_per_active_{minutes}m'] = (
            out['events_ext'] / out[f'active_bins_{minutes}m'].replace(0, np.nan)
        )

    for window in [300, 1800, 7200]:
        edge = w.assign(
            _edge=((w['rel_sec'] <= window) | (w['rel_sec'] >= 86400 - window)).astype(int)
        ).groupby('cookie_id')['_edge'].mean().rename(f'edge_activity_share_{window}s')
        out = out.merge(edge, on='cookie_id', how='left')

    out = out.merge(
        w.assign(_h=(w['rel_sec'] < 43200).astype(int))
         .groupby('cookie_id')['_h'].mean().rename('first_half_share'),
        on='cookie_id', how='left'
    )
    out = out.merge(
        w.assign(_h=(w['rel_sec'] >= 43200).astype(int))
         .groupby('cookie_id')['_h'].mean().rename('second_half_share'),
        on='cookie_id', how='left'
    )

    # Распределение активности по часам.
    hour_counts = w.groupby(['cookie_id', 'hour']).size().rename('n').reset_index()
    hour_p = hour_counts['n'] / hour_counts.groupby('cookie_id')['n'].transform('sum')
    hour_entropy = (
        -hour_p * np.log2(np.clip(hour_p, 1e-12, None))
    ).groupby(hour_counts['cookie_id']).sum().rename('hour_entropy')
    hour_top = (
        hour_counts.groupby('cookie_id')['n'].max() /
        hour_counts.groupby('cookie_id')['n'].sum()
    ).rename('peak_hour_share')
    hour_unique = hour_counts.groupby('cookie_id')['hour'].nunique().rename('active_hours_count')
    out = out.merge(hour_entropy, on='cookie_id', how='left')
    out = out.merge(hour_top, on='cookie_id', how='left')
    out = out.merge(hour_unique, on='cookie_id', how='left')

    # Паузы: квантили, короткие/длинные интервалы, burstiness, регулярность.
    gap_valid = w.dropna(subset=['gap_sec']).copy()
    if not gap_valid.empty:
        gap_q = gap_valid.groupby('cookie_id')['gap_sec'].agg(
            gap_q01=lambda x: x.quantile(.01),
            gap_q05=lambda x: x.quantile(.05),
            gap_q10=lambda x: x.quantile(.10),
            gap_q25=lambda x: x.quantile(.25),
            gap_q75=lambda x: x.quantile(.75),
            gap_q90=lambda x: x.quantile(.90),
            gap_q95=lambda x: x.quantile(.95),
            gap_q99=lambda x: x.quantile(.99),
        ).reset_index()
        out = out.merge(gap_q, on='cookie_id', how='left')

        for thr in [1, 2, 5, 10, 30, 60]:
            s = gap_valid.groupby('cookie_id')['gap_sec'].apply(
                lambda x, t=thr: (x <= t).mean()
            ).rename(f'gap_le_{thr}s')
            out = out.merge(s, on='cookie_id', how='left')
        for thr in [300, 900, 3600]:
            s = gap_valid.groupby('cookie_id')['gap_sec'].apply(
                lambda x, t=thr: (x > t).mean()
            ).rename(f'gap_gt_{thr}s')
            out = out.merge(s, on='cookie_id', how='left')

        gap_valid['gap_log_bin'] = np.floor(
            np.log1p(np.clip(gap_valid['gap_sec'], 0, None))
        ).astype(int)
        gap_bins = (
            gap_valid.groupby(['cookie_id', 'gap_log_bin']).size()
            .rename('n').reset_index()
        )
        gp = gap_bins['n'] / gap_bins.groupby('cookie_id')['n'].transform('sum')
        gap_entropy = (
            -gp * np.log2(np.clip(gp, 1e-12, None))
        ).groupby(gap_bins['cookie_id']).sum().rename('gap_log_entropy')
        out = out.merge(gap_entropy, on='cookie_id', how='left')

        gap_stats = gap_valid.groupby('cookie_id')['gap_sec'].agg(['mean', 'std'])
        gap_stats['gap_burstiness'] = (
            (gap_stats['std'] - gap_stats['mean']) /
            (gap_stats['std'] + gap_stats['mean'] + 1e-9)
        )
        gap_stats['gap_regularity'] = 1 / (
            1 + gap_stats['std'] / gap_stats['mean'].replace(0, np.nan)
        )
        out = out.merge(
            gap_stats[['gap_burstiness', 'gap_regularity']].reset_index(),
            on='cookie_id', how='left'
        )
        out = out.merge(
            gap_valid.groupby('cookie_id')['gap_sec']
            .apply(lambda x: (x <= 0).mean())
            .rename('zero_gap_share'),
            on='cookie_id', how='left'
        )

    # Сессионный ритм: >30 минут = новая сессия.
    w['new_session_30m'] = (w['gap_sec'].isna() | (w['gap_sec'] > 1800)).astype(int)
    w['session_no'] = w.groupby('cookie_id')['new_session_30m'].cumsum()
    sess = (
        w.groupby(['cookie_id', 'session_no'])
         .agg(sess_start=('event_ts', 'min'), sess_end=('event_ts', 'max'), sess_events=('event_ts', 'size'))
         .reset_index()
    )
    sess['sess_duration_sec'] = (
        sess['sess_end'] - sess['sess_start']
    ).dt.total_seconds().fillna(0)
    sess_sum = (
        sess.groupby('cookie_id')
        .agg(
            session_count=('session_no', 'nunique'),
            session_duration_max=('sess_duration_sec', 'max'),
            session_duration_median=('sess_duration_sec', 'median'),
            session_duration_mean=('sess_duration_sec', 'mean'),
            session_events_max=('sess_events', 'max'),
            session_events_median=('sess_events', 'median'),
            session_events_mean=('sess_events', 'mean'),
            session_events_std=('sess_events', 'std'),
        ).reset_index()
    )
    sess_sum['session_events_cv'] = (
        sess_sum['session_events_std'] /
        sess_sum['session_events_mean'].replace(0, np.nan)
    )
    out = out.merge(sess_sum, on='cookie_id', how='left')
    sess_starts = sess.sort_values(['cookie_id', 'sess_start'])
    inter_session_gap = sess_starts.groupby('cookie_id')['sess_start'].diff().dt.total_seconds()
    out = out.merge(
        inter_session_gap.groupby(sess_starts['cookie_id']).mean().rename('inter_session_gap_mean'),
        left_on='cookie_id', right_index=True, how='left'
    )

    # Sliding burst density.
    burst_rows = []
    for cid, gg in w[['cookie_id', 'rel_sec']].groupby('cookie_id', sort=False):
        vals = np.sort(gg['rel_sec'].to_numpy(dtype=float))
        row = {'cookie_id': cid}
        for sec in [10, 30, 60, 300, 900, 3600]:
            row[f'max_events_{sec}s'] = _max_events_in_window(vals, sec)
        burst_rows.append(row)
    out = out.merge(pd.DataFrame(burst_rows), on='cookie_id', how='left')

    # Дискретные распределения: повторяемость + энтропия.
    for col, prefix in [
        ('event_name', 'event'), ('platform', 'platform'), ('user_agent', 'ua'),
        ('ua_os', 'os'), ('ua_browser', 'browser'), ('item_id', 'item'),
        ('item_category', 'category'), ('item_location', 'location'),
        ('seller_type', 'seller'), ('search_query', 'query'),
    ]:
        if col in w.columns:
            out = out.merge(_distribution_metrics(w, col, prefix), on='cookie_id', how='left')

    out = out.merge(
        w.groupby('cookie_id')['event_change'].mean().rename('event_change_rate'),
        on='cookie_id', how='left'
    )
    for col in ['platform_switch', 'user_agent_switch', 'ua_browser_switch', 'ua_os_switch']:
        if col in w.columns:
            out = out.merge(
                w.groupby('cookie_id')[col].mean().rename(col + '_rate'),
                on='cookie_id', how='left'
            )

    # Последовательностные паттерны событий: боты часто повторяют одинаковые переходы
    # и короткие циклы. Считаем только по событиям внутри наблюдаемого окна.
    seq_rows = []
    for cid, gg in w[['cookie_id', 'event_name']].groupby('cookie_id', sort=False):
        seq = gg['event_name'].astype(str).to_numpy()
        row = {'cookie_id': cid, 'same_event_repeat_share': 0.0, 'max_same_event_run': 1 if len(seq) else 0,
               'unique_transitions': 0, 'top_transition_share': 0.0, 'transition_entropy': 0.0}
        if len(seq) >= 2:
            same = seq[1:] == seq[:-1]
            row['same_event_repeat_share'] = float(np.mean(same))
            run = 1; max_run = 1
            for flag in same:
                run = run + 1 if flag else 1
                max_run = max(max_run, run)
            row['max_same_event_run'] = max_run
            pairs = pd.Series(list(zip(seq[:-1], seq[1:])))
            pc = pairs.value_counts()
            row['unique_transitions'] = int(pc.size)
            row['top_transition_share'] = float(pc.iloc[0] / pc.sum())
            pp = pc.to_numpy(dtype=float) / pc.sum()
            row['transition_entropy'] = float(-(pp * np.log2(np.clip(pp, 1e-12, None))).sum())
        seq_rows.append(row)
    out = out.merge(pd.DataFrame(seq_rows), on='cookie_id', how='left')

    # Время до первого события и idle-пауза перед концом окна. Это особенно полезно
    # для различения cookie, которые сразу начинают плотный машинный сбор, от обычного browsing.
    edge_times = timing[['cookie_id', 'first_event', 'last_event']].copy()
    meta_for_edge = meta_small.set_index('cookie_id')
    edge_times['start_delay_sec'] = (
        edge_times['first_event'] - edge_times['cookie_id'].map(meta_for_edge['window_start_ts'])
    ).dt.total_seconds().clip(lower=0)
    edge_times['end_idle_sec'] = (
        edge_times['cookie_id'].map(meta_for_edge['window_end_ts']) - edge_times['last_event']
    ).dt.total_seconds().clip(lower=0)
    out = out.merge(edge_times[['cookie_id', 'start_delay_sec', 'end_idle_sec']], on='cookie_id', how='left')

    # --- Shared fingerprints inside the same observation day ---
    # Это не target-derived информация: для каждой cookie считаем, насколько часто
    # тот же UA/item/query встречается у ДРУГИХ cookie в том же окне. Такой сигнал
    # может ловить фермы/агрегаторы, использующие повторяющуюся инфраструктуру.
    def add_peer_feature(col, prefix):
        if col not in w.columns:
            return
        base = w[['cookie_id', 'window_start_ts', col]].dropna(subset=[col]).copy()
        if base.empty:
            return
        peers = (
            base.groupby(['window_start_ts', col])['cookie_id']
            .nunique()
            .rename('_peer_cookies')
            .reset_index()
        )
        base = base.merge(peers, on=['window_start_ts', col], how='left')
        base['_other_cookie_count'] = (base['_peer_cookies'] - 1).clip(lower=0)
        agg = base.groupby('cookie_id')['_other_cookie_count'].agg(
            **{f'{prefix}_other_cookie_mean': 'mean',
               f'{prefix}_other_cookie_max': 'max',
               f'{prefix}_shared_event_share': lambda x: float((x > 0).mean())}
        ).reset_index()
        out_key = prefix + '_other_cookie_mean'
        nonlocal_out[0] = nonlocal_out[0].merge(agg, on='cookie_id', how='left')

    nonlocal_out = [out]
    for col, prefix in [
        ('user_agent', 'ua_peer'),
        ('item_id', 'item_peer'),
        ('search_query', 'query_peer'),
    ]:
        add_peer_feature(col, prefix)
    out = nonlocal_out[0]

    # Поиск: не только query_diversity, но и глубина / хвост выдачи.
    search = w[w['search_query'].notna()].copy()
    if not search.empty:
        sp = search.groupby('cookie_id')['search_page'].agg(
            search_page_mean='mean', search_page_std='std', search_page_max='max'
        ).reset_index()
        sp['search_page_high_share'] = (
            search.groupby('cookie_id')['search_page'].apply(lambda x: (x >= 3).mean()).to_numpy()
        )
        sp['search_page_very_high_share'] = (
            search.groupby('cookie_id')['search_page'].apply(lambda x: (x >= 5).mean()).to_numpy()
        )
        out = out.merge(sp, on='cookie_id', how='left')

    contact_names = {'contact_phone_show', 'contact_chat_open', 'contact_message_sent'}
    w['is_contact'] = w['event_name'].isin(contact_names).astype(int)
    w['is_engagement'] = w['event_name'].isin({'favorite_add', 'seller_page_view'}).astype(int)
    out = out.merge(
        w.groupby('cookie_id')['is_contact'].agg(contact_events='sum', contact_share='mean'),
        on='cookie_id', how='left'
    )
    out = out.merge(
        w.groupby('cookie_id')['is_engagement'].agg(engagement_events='sum', engagement_share='mean'),
        on='cookie_id', how='left'
    )

    # Pointer trajectory: path length / step length / speed / straightness.
    pointer_rows = []
    for cid, gg in w.dropna(subset=['pointer_x', 'pointer_y']).groupby('cookie_id', sort=False):
        gg = gg.sort_values('event_ts')
        x = gg['pointer_x'].to_numpy(float)
        y = gg['pointer_y'].to_numpy(float)
        t = gg['event_ts'].astype('int64').to_numpy() / 1e9
        row = {'cookie_id': cid, 'pointer_steps': max(len(x) - 1, 0)}
        if len(x) >= 2:
            dx, dy = np.diff(x), np.diff(y)
            dist = np.hypot(dx, dy)
            dt = np.diff(t)
            valid_dt = dt > 0
            speed = dist[valid_dt] / dt[valid_dt]
            path = float(dist.sum())
            direct = float(np.hypot(x[-1] - x[0], y[-1] - y[0]))
            row.update({
                'pointer_path_length': path,
                'pointer_step_mean': float(np.mean(dist)),
                'pointer_step_median': float(np.median(dist)),
                'pointer_step_std': float(np.std(dist)),
                'pointer_zero_step_share': float(np.mean(dist == 0)),
                'pointer_path_efficiency': float(direct / (path + 1e-9)),
                'pointer_speed_mean': float(np.mean(speed)) if len(speed) else 0.0,
                'pointer_speed_median': float(np.median(speed)) if len(speed) else 0.0,
                'pointer_speed_p95': float(np.quantile(speed, .95)) if len(speed) else 0.0,
            })
        else:
            row.update({
                'pointer_path_length': 0.0, 'pointer_step_mean': 0.0,
                'pointer_step_median': 0.0, 'pointer_step_std': 0.0,
                'pointer_zero_step_share': 0.0, 'pointer_path_efficiency': 0.0,
                'pointer_speed_mean': 0.0, 'pointer_speed_median': 0.0,
                'pointer_speed_p95': 0.0,
            })
        pointer_rows.append(row)
    if pointer_rows:
        out = out.merge(pd.DataFrame(pointer_rows), on='cookie_id', how='left')

    # Простые отношения — особенно полезны деревьям.
    if 'item_count' in out:
        out['items_per_event_ext'] = out['item_count'] / out['events_ext'].replace(0, np.nan)
    if 'category_count' in out:
        out['categories_per_event_ext'] = out['category_count'] / out['events_ext'].replace(0, np.nan)
    if 'location_count' in out:
        out['locations_per_event_ext'] = out['location_count'] / out['events_ext'].replace(0, np.nan)
    if 'query_count' in out:
        out['queries_per_event_ext'] = out['query_count'] / out['events_ext'].replace(0, np.nan)
    if 'pointer_count' in out:
        out['pointer_per_event_ext'] = out['pointer_count'] / out['events_ext'].replace(0, np.nan)

    num_cols = [c for c in out.columns if c != 'cookie_id']
    out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan)
    return out.fillna(0)


Собираю итоговый BASE и EXTENDED наборы.

In [30]:


meta_for_features = pd.concat([
    train[['cookie_id', 'window_start_ts', 'window_end_ts']],
    test[['cookie_id', 'window_start_ts', 'window_end_ts']],
], ignore_index=True).drop_duplicates('cookie_id')

extended_features = build_extended_behavior_features(events, meta_for_features)
new_extended_cols = [
    c for c in extended_features.columns
    if c != 'cookie_id' and c not in BASE_COOKIE_FEATURES
]
cookie_dataset = cookie_dataset.merge(
    extended_features[['cookie_id'] + new_extended_cols],
    on='cookie_id', how='left', validate='one_to_one'
)

print(f'BASE признаков (до новых behavioral): {len(BASE_COOKIE_FEATURES)-1}')
print(f'Новых behavioral-признаков: {len(new_extended_cols)}')
print(f'Итого признаков в cookie_dataset: {cookie_dataset.shape[1]-1}')

BASE признаков (до новых behavioral): 57
Новых behavioral-признаков: 161
Итого признаков в cookie_dataset: 218


## 2. Матрицы признаков
Разделяю BASE и EXTENDED наборы.

Готовлю train/test матрицы BASE и EXTENDED.

In [31]:
all_ids = pd.concat([train[['cookie_id']], test[['cookie_id']]], ignore_index=True)
missing_ids = set(all_ids['cookie_id']) - set(cookie_dataset['cookie_id'])
print(f'cookie_id без события в окне: {len(missing_ids)} из {len(all_ids)}')

CAT_FEATURES = ['main_platform', 'main_os', 'main_browser', 'main_category']

df_train = train[['cookie_id', 'window_start_ts', 'cookie_created_at', 'target']].merge(
    all_ids.merge(cookie_dataset, on='cookie_id', how='left'),
    on='cookie_id', how='left'
)
df_test = test[['cookie_id', 'window_start_ts', 'cookie_created_at']].merge(
    all_ids.merge(cookie_dataset, on='cookie_id', how='left'),
    on='cookie_id', how='left'
)

# На случай любых внутренних преобразований гарантируем полный test/train coverage.
assert len(df_train) == len(train)
assert len(df_test) == len(test)
assert df_train['cookie_id'].is_unique and df_test['cookie_id'].is_unique
assert set(df_test['cookie_id']) == set(test['cookie_id'])

# Один и тот же набор категорий для train и test.
for c in CAT_FEATURES:
    all_cat = pd.concat([
        df_train[c].astype('string'), df_test[c].astype('string')
    ], ignore_index=True).fillna('no_events').astype('category')
    cats = all_cat.cat.categories
    df_train[c] = pd.Categorical(df_train[c].astype('string').fillna('no_events'), categories=cats)
    df_test[c] = pd.Categorical(df_test[c].astype('string').fillna('no_events'), categories=cats)

ID_COLS = ['cookie_id', 'window_start_ts', 'cookie_created_at', 'target', 'event_day']
BASE_NUM_FEATURES = [
    c for c in BASE_COOKIE_FEATURES
    if c not in ['cookie_id', 'event_day'] + CAT_FEATURES
]
EXT_NUM_FEATURES = [
    c for c in df_train.columns
    if c not in ID_COLS + CAT_FEATURES
]

X_train_base = df_train[BASE_NUM_FEATURES + CAT_FEATURES].copy()
X_test_base = df_test[BASE_NUM_FEATURES + CAT_FEATURES].copy()
X_train_ext = df_train[EXT_NUM_FEATURES + CAT_FEATURES].copy()
X_test_ext = df_test[EXT_NUM_FEATURES + CAT_FEATURES].copy()
y = df_train['target'].to_numpy(dtype=int)

print(f'BASE: {len(BASE_NUM_FEATURES)} numeric + {len(CAT_FEATURES)} categorical')
print(f'EXTENDED: {len(EXT_NUM_FEATURES)} numeric + {len(CAT_FEATURES)} categorical')
print(f'bot rate: {y.mean():.4f}')

cookie_id без события в окне: 0 из 16000
BASE: 52 numeric + 4 categorical
EXTENDED: 213 numeric + 4 categorical
bot rate: 0.0811


## 3. Валидация по времени
Оставляю исходные разрезы 0.6 / 0.7 / 0.8.

Создаю временные фолды.

In [32]:
from metric import precision_at_recall
from sklearn.metrics import roc_auc_score, average_precision_score


def time_splits(ts, quantiles=(0.6, 0.7, 0.8)):
    ts = pd.Series(ts).reset_index(drop=True)
    out = []
    for q in quantiles:
        cutoff = ts.quantile(q)
        va_mask = (ts >= cutoff).to_numpy()
        tr_mask = ~va_mask
        if tr_mask.sum() > 500 and va_mask.sum() > 200:
            out.append((tr_mask, va_mask))
    return out

RANDOM_STATE = 0
splits = time_splits(df_train['window_start_ts'])

print('Фолды:')
for i, (tr_mask, va_mask) in enumerate(splits, 1):
    print(
        f'  fold {i}: train={tr_mask.sum():,}, valid={va_mask.sum():,}, '
        f'bots={y[va_mask].sum():,}, bot_rate={y[va_mask].mean():.3f}'
    )

Фолды:
  fold 1: train=5,930, valid=5,161, bots=408, bot_rate=0.079
  fold 2: train=7,599, valid=3,492, bots=285, bot_rate=0.082
  fold 3: train=8,400, valid=2,691, bots=220, bot_rate=0.082


## 5. Новые кандидаты моделей
Сравниваю несколько моделей и бустингов.

In [33]:
# ============================================================
# AverageEnsemble
# ============================================================
# Универсальный soft-voting ансамбль.
#
# ВАЖНО:
# factories — это функции, которые при вызове создают новую
# необученную модель.
#
# Поэтому каждый компонент ансамбля независимо обучается
# на том же train-фолде, после чего усредняются вероятности.
# ============================================================

class AverageEnsemble:
    def __init__(self, factories, weights=None):
        self.factories = list(factories)

        if weights is None:
            weights = np.ones(len(self.factories), dtype=float)

        self.weights = np.asarray(weights, dtype=float)

        if len(self.weights) != len(self.factories):
            raise ValueError(
                "Количество weights должно совпадать "
                "с количеством моделей."
            )

        if np.any(self.weights < 0):
            raise ValueError("Weights должны быть >= 0.")

        if self.weights.sum() <= 0:
            raise ValueError("Сумма weights должна быть > 0.")

        self.weights = self.weights / self.weights.sum()
        self.models_ = []

    def fit(self, X, y):
        self.models_ = []

        for i, factory in enumerate(self.factories):
            model = factory()

            print(
                f"  ensemble member {i + 1}/{len(self.factories)} "
                f"(weight={self.weights[i]:.3f})"
            )

            model.fit(X, y)
            self.models_.append(model)

        return self

    def predict_proba(self, X):
        if not self.models_:
            raise RuntimeError(
                "Ensemble ещё не обучен. Сначала вызови fit()."
            )

        probabilities = np.zeros(len(X), dtype=float)

        for weight, model in zip(self.weights, self.models_):
            probabilities += (
                weight * model.predict_proba(X)[:, 1]
            )

        # Возвращаем стандартный формат sklearn:
        # [:, 0] = P(class 0)
        # [:, 1] = P(class 1)
        return np.column_stack([
            1.0 - probabilities,
            probabilities
        ])

Подключаю библиотеки для моделей.

In [34]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ---------- Preprocessors ----------


Готовлю препроцессинг для разных моделей.

In [35]:
def make_lr(num_cols, X_ref):
    # log1p только для неотрицательных skewed-признаков; это вычисляется по train без target.
    skewed = []
    for c in num_cols:
        s = pd.to_numeric(X_ref[c], errors='coerce')
        if s.notna().sum() >= 10 and s.min(skipna=True) >= 0 and abs(s.skew()) > 1.0:
            skewed.append(c)
    other = [c for c in num_cols if c not in skewed]
    prep = ColumnTransformer([
        ('log_num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
            ('scale', StandardScaler()),
        ]), skewed),
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), other),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), CAT_FEATURES),
    ])
    return Pipeline([
        ('prep', prep),
        ('clf', LogisticRegression(
            max_iter=2500, class_weight='balanced', C=0.5, random_state=RANDOM_STATE
        )),
    ])


def make_tree_bagging(model_cls, **params):
    prep = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES)
    ], remainder='passthrough')
    return Pipeline([
        ('prep', prep),
        ('impute', SimpleImputer(strategy='median')),
        ('clf', model_cls(**params)),
    ])


def make_hgb(num_cols, **params):
    prep = ColumnTransformer([
        ('cat', OrdinalEncoder(
            handle_unknown='use_encoded_value', unknown_value=-1,
            encoded_missing_value=-1
        ), CAT_FEATURES)
    ], remainder='passthrough', verbose_feature_names_out=False)
    cat_mask = [True] * len(CAT_FEATURES) + [False] * len(num_cols)
    return Pipeline([
        ('prep', prep),
        ('clf', HistGradientBoostingClassifier(
            categorical_features=cat_mask,
            class_weight='balanced', random_state=RANDOM_STATE,
            **params
        )),
    ])




Настраиваю LightGBM и CatBoost.

In [36]:
class LGBMWrapper:
    """LightGBM с одинаковыми categorical columns для train/valid/test."""
    def __init__(self, params):
        self.params = params

    def _prepare(self, X):
        X = X.copy()
        for c in CAT_FEATURES:
            X[c] = X[c].astype('category')
        return X

    def fit(self, X, y):
        self.model_ = LGBMClassifier(**self.params)
        Xp = self._prepare(X)
        self.model_.fit(Xp, y, categorical_feature=CAT_FEATURES)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(self._prepare(X))


class CatBoostWrapper:
    """CatBoost с явной передачей категориальных колонок."""
    def __init__(self, params):
        self.params = params

    def _prepare(self, X):
        X = X.copy()
        for c in CAT_FEATURES:
            X[c] = X[c].astype('string').fillna('__NA__')
        return X

    def fit(self, X, y):
        self.model_ = CatBoostClassifier(**self.params)
        Xp = self._prepare(X)
        self.model_.fit(Xp, y, cat_features=CAT_FEATURES)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(self._prepare(X))


# ---------- Модельные фабрики ----------
# EXTENDED HistGB: чуть сильнее/регуляризованнее старой конфигурации.


Определяю фабрики моделей.

In [37]:
def ext_hgb():
    return make_hgb(
        EXT_NUM_FEATURES,
        max_iter=400, learning_rate=0.04, max_depth=6,
        min_samples_leaf=25, l2_regularization=2.0
    )


def base_hgb():
    return make_hgb(
        BASE_NUM_FEATURES,
        max_iter=350, learning_rate=0.04, max_depth=6,
        min_samples_leaf=25, l2_regularization=2.0
    )


def base_rf():
    return make_tree_bagging(
        RandomForestClassifier,
        n_estimators=600, min_samples_leaf=3, max_features='sqrt',
        class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1
    )


def base_extra():
    return make_tree_bagging(
        ExtraTreesClassifier,
        n_estimators=700, min_samples_leaf=2, max_features='sqrt',
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    )


def lgbm_recipe(params):
    return lambda: LGBMWrapper({
        'objective': 'binary',
        'colsample_bytree': 0.90,
        'subsample': 0.90,
        'subsample_freq': 1,
        'class_weight': 'balanced',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbosity': -1,
        **params,
    })


def cat_recipe(params):
    return lambda: CatBoostWrapper({
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'auto_class_weights': 'Balanced',
        'random_seed': RANDOM_STATE,
        'thread_count': -1,
        'verbose': False,
        'allow_writing_files': False,
        **params,
    })


# Эти рецепты специально включают и уже проверенные значения, и более регуляризованные варианты.


Собираю список кандидатов.

In [38]:
LGBM_RECIPES = {
    'EXT_LGBM_A': dict(n_estimators=700, learning_rate=0.03, num_leaves=31, min_child_samples=35, reg_alpha=0.25, reg_lambda=4.0),
    'EXT_LGBM_B': dict(n_estimators=700, learning_rate=0.03, num_leaves=63, min_child_samples=60, reg_alpha=0.35, reg_lambda=8.0),
    'EXT_LGBM_C': dict(n_estimators=900, learning_rate=0.02, num_leaves=31, min_child_samples=50, reg_alpha=0.25, reg_lambda=6.0),
    'EXT_LGBM_D': dict(n_estimators=850, learning_rate=0.025, num_leaves=47, min_child_samples=70, reg_alpha=0.5, reg_lambda=10.0),
}

CAT_RECIPES = {
    'EXT_CAT_A': dict(iterations=800, depth=6, learning_rate=0.04, l2_leaf_reg=8.0, random_strength=0.5),
    'EXT_CAT_B': dict(iterations=900, depth=7, learning_rate=0.03, l2_leaf_reg=10.0, random_strength=1.0),
    'EXT_CAT_C': dict(iterations=700, depth=5, learning_rate=0.045, l2_leaf_reg=6.0, random_strength=0.3),
}

MODEL_SPECS = {
    'BASE_LogReg': ('base', lambda: make_lr(BASE_NUM_FEATURES, X_train_base)),
    'BASE_RF': ('base', base_rf),
    'BASE_ExtraTrees': ('base', base_extra),
    'BASE_HistGB': ('base', base_hgb),
    'EXT_HistGB': ('extended', ext_hgb),
    # Контрольная LightGBM-конфигурация из предыдущей рабочей ветки.
    'BASE_LGBM_A': ('base', lgbm_recipe(LGBM_RECIPES['EXT_LGBM_A'])),
    'BASE_LGBM_B': ('base', lgbm_recipe(LGBM_RECIPES['EXT_LGBM_B'])),
    'BASE_CAT_A': ('base', cat_recipe(CAT_RECIPES['EXT_CAT_A'])),
    'EXT_CAT_A': ('extended', cat_recipe(CAT_RECIPES['EXT_CAT_A'])),
    'EXT_CAT_B': ('extended', cat_recipe(CAT_RECIPES['EXT_CAT_B'])),
    'EXT_CAT_C': ('extended', cat_recipe(CAT_RECIPES['EXT_CAT_C'])),
    # Контрольный ансамбль из предыдущего решения: LGBM + RF + HistGB.
    'BASE_Ensemble(LGBM+RF+HistGB)': ('base',
        lambda: AverageEnsemble([
            lgbm_recipe(LGBM_RECIPES['EXT_LGBM_A']), base_rf, base_hgb
        ], weights=[1.5, 1.0, 1.0])
    ),
}
for name, params in LGBM_RECIPES.items():
    MODEL_SPECS[name] = ('extended', lgbm_recipe(params))

X_BY_SET = {
    'base': X_train_base,
    'extended': X_train_ext,
}

## 4. Старый стек базовых признаках
Модели, которые обучались после первого варианта предобработки данных (до добавления 200 признаков). На них был получен первый submission с метрикой, равной 0.72

### Старый набор 52 признаков
Восстанавливаю матрицу из предыдущего ноутбука отдельно от финальной ветки.

In [39]:
# Восстанавливаю ровно тот старый набор, который использовался до extended-признаков.
OLD52_CAT_FEATURES = ['main_platform', 'main_os', 'main_browser', 'main_category']
OLD52_RAW_COLUMNS = [c for c in BASE_COOKIE_FEATURES_RAW if c != 'cookie_id']

OLD52_COOKIE_DATASET = cookie_dataset[['cookie_id'] + OLD52_RAW_COLUMNS].copy()

OLD52_ALL_IDS = pd.concat(
    [train[['cookie_id']], test[['cookie_id']]],
    ignore_index=True
)

OLD52_COOKIE_DATASET_FULL = OLD52_ALL_IDS.merge(
    OLD52_COOKIE_DATASET,
    on='cookie_id',
    how='left'
)

OLD52_COOKIE_DATASET_FULL['had_events'] = (
    OLD52_COOKIE_DATASET_FULL['session_duration_sec'].notna().astype(int)
)

for c in OLD52_CAT_FEATURES:
    OLD52_COOKIE_DATASET_FULL[c] = OLD52_COOKIE_DATASET_FULL[c].fillna('no_events')

OLD52_TRAIN = train[
    ['cookie_id', 'window_start_ts', 'cookie_created_at', 'target']
].merge(
    OLD52_COOKIE_DATASET_FULL,
    on='cookie_id',
    how='left'
)

OLD52_TEST = test[
    ['cookie_id', 'window_start_ts', 'cookie_created_at']
].merge(
    OLD52_COOKIE_DATASET_FULL,
    on='cookie_id',
    how='left'
)

for d in (OLD52_TRAIN, OLD52_TEST):
    d['account_age_days'] = (
        d['window_start_ts'] - d['cookie_created_at']
    ).dt.total_seconds() / 86400

OLD52_ID_COLS = [
    'cookie_id', 'window_start_ts', 'cookie_created_at',
    'target', 'event_day'
]

OLD52_NUM_FEATURES = [
    c for c in OLD52_TRAIN.columns
    if c not in OLD52_ID_COLS + OLD52_CAT_FEATURES
]

OLD52_X_TRAIN = OLD52_TRAIN[
    OLD52_NUM_FEATURES + OLD52_CAT_FEATURES
].copy()

OLD52_X_TEST = OLD52_TEST[
    OLD52_NUM_FEATURES + OLD52_CAT_FEATURES
].copy()

assert len(OLD52_X_TRAIN) == len(train)
assert len(OLD52_X_TEST) == len(test)
assert set(OLD52_TRAIN['cookie_id']) == set(train['cookie_id'])
assert set(OLD52_TEST['cookie_id']) == set(test['cookie_id'])
assert OLD52_X_TRAIN.shape[1] == 52
assert len(OLD52_NUM_FEATURES) == 48

print(
    f'OLD52: {len(OLD52_NUM_FEATURES)} numeric + '
    f'{len(OLD52_CAT_FEATURES)} categorical = '
    f'{OLD52_X_TRAIN.shape[1]} features'
)


OLD52: 48 numeric + 4 categorical = 52 features


### Оценка старых моделей
Использую ту же официальную метрику и те же временные фолды.

In [40]:
import warnings

def evaluate_old_models(model_dict, X_ref, y_ref, split_ref):
    rows = []

    for name, factory in model_dict.items():
        p_scores = []
        roc_scores = []
        pr_scores = []

        for tr_mask, va_mask in split_ref:
            model = factory()
            model.fit(X_ref.loc[tr_mask], y_ref[tr_mask])

            p_va = model.predict_proba(
                X_ref.loc[va_mask]
            )[:, 1]

            y_va = y_ref[va_mask]

            p_scores.append(
                precision_at_recall(y_va, p_va)
            )
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                roc_scores.append(
                    roc_auc_score(y_va, p_va)
                )
                pr_scores.append(
                    average_precision_score(y_va, p_va)
                )

        rows.append({
            'model': name,
            'P@R0.7_mean': float(np.mean(p_scores)),
            'P@R0.7_std': float(np.std(p_scores)),
            'P@R0.7_last': float(p_scores[-1]),
            'ROC-AUC': float(np.mean(roc_scores)),
            'PR-AUC': float(np.mean(pr_scores)),
        })

    return pd.DataFrame(rows).sort_values(
        ['P@R0.7_mean', 'P@R0.7_last'],
        ascending=False
    ).reset_index(drop=True)


### Старый блок с gap-признаками
В старой рабочей ветке к 52 базовым признакам добавлялись только 5 gap-признаков.

### Старый набор с gap
Добавляю только пять старых timing-признаков и снова сравниваю модели.

In [41]:
OLD52_GAP_COLS = [
    'gap_mean', 'gap_std', 'gap_median',
    'gap_min', 'gap_cv'
]

gap_lookup_train = df_train.set_index('cookie_id')[OLD52_GAP_COLS]
gap_lookup_test = df_test.set_index('cookie_id')[OLD52_GAP_COLS]

OLD52_TRAIN_GAP = OLD52_TRAIN.merge(
    gap_lookup_train,
    left_on='cookie_id',
    right_index=True,
    how='left',
    validate='one_to_one'
)

OLD52_TEST_GAP = OLD52_TEST.merge(
    gap_lookup_test,
    left_on='cookie_id',
    right_index=True,
    how='left',
    validate='one_to_one'
)

OLD52_GAP_NUM_FEATURES = [
    c for c in OLD52_TRAIN_GAP.columns
    if c not in OLD52_ID_COLS + OLD52_CAT_FEATURES
]

OLD52_GAP_X_TRAIN = OLD52_TRAIN_GAP[
    OLD52_GAP_NUM_FEATURES + OLD52_CAT_FEATURES
].copy()

OLD52_GAP_X_TEST = OLD52_TEST_GAP[
    OLD52_GAP_NUM_FEATURES + OLD52_CAT_FEATURES
].copy()

assert OLD52_GAP_X_TRAIN.shape[1] == 57
assert len(OLD52_GAP_X_TEST) == len(test)

print(
    f'OLD52 + gaps: {len(OLD52_GAP_NUM_FEATURES)} numeric + '
    f'{len(OLD52_CAT_FEATURES)} categorical = '
    f'{OLD52_GAP_X_TRAIN.shape[1]} features'
)


OLD52 + gaps: 53 numeric + 4 categorical = 57 features


### Старый ансамбль с gap
результат около 0.72 без новых extended-признаков.

In [42]:
OLD52_GAP_SKEWED_NUM = [c for c in [
    'session_duration_sec', 'events_count', 'total_item_views', 'unique_items_count',
    'search_count', 'pointer_events_count', 'non_megapolis_views_count',
    'unique_search_queries_count', 'unique_locations_count', 'unique_categories_count',
    'gap_mean', 'gap_median'
] if c in OLD52_GAP_NUM_FEATURES]

OLD52_GAP_OTHER_NUM = [
    c for c in OLD52_GAP_NUM_FEATURES
    if c not in OLD52_GAP_SKEWED_NUM
]

OLD52_GAP_PREPROCESS_LR = ColumnTransformer([
    ('log_num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('scale', StandardScaler()),
    ]), OLD52_GAP_SKEWED_NUM),
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), OLD52_GAP_OTHER_NUM),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('ohe', OneHotEncoder(handle_unknown='ignore')),
    ]), OLD52_CAT_FEATURES),
])

OLD52_GAP_PREPROCESS_HGB = ColumnTransformer(
    transformers=[('cat', OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1,
        encoded_missing_value=-1
    ), OLD52_CAT_FEATURES)],
    remainder='passthrough',
    verbose_feature_names_out=False
)

OLD52_GAP_CATEGORICAL_MASK = (
    [True] * len(OLD52_CAT_FEATURES)
    + [False] * len(OLD52_GAP_NUM_FEATURES)
)

def old52_gap_build_lr():
    return Pipeline([
        ('prep', OLD52_GAP_PREPROCESS_LR),
        ('clf', LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            C=0.5,
            random_state=RANDOM_STATE
        )),
    ])

def old52_gap_build_rf():
    return Pipeline([
        ('prep', ColumnTransformer([
            ('cat', OneHotEncoder(
                handle_unknown='ignore'
            ), OLD52_CAT_FEATURES)
        ], remainder='passthrough')),
        ('impute', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=3,
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE,
            n_jobs=-1
        )),
    ])

def old52_gap_build_hgb():
    return Pipeline([
        ('prep', OLD52_GAP_PREPROCESS_HGB),
        ('clf', HistGradientBoostingClassifier(
            categorical_features=OLD52_GAP_CATEGORICAL_MASK,
            max_iter=300,
            learning_rate=0.05,
            max_depth=6,
            min_samples_leaf=30,
            l2_regularization=1.0,
            class_weight='balanced',
            random_state=RANDOM_STATE
        )),
    ])

OLD52_GAP_MODELS = {
    'OLD52_GAP_LogReg': old52_gap_build_lr,
    'OLD52_GAP_RandomForest': old52_gap_build_rf,
    'OLD52_GAP_HistGB': old52_gap_build_hgb,
    'OLD52_GAP_Ensemble(LR+RF+HistGB)': lambda: AverageEnsemble(
        [old52_gap_build_lr, old52_gap_build_rf, old52_gap_build_hgb]
    ),
}


In [43]:
OLD52_GAP_RESULTS = evaluate_old_models(
    OLD52_GAP_MODELS,
    OLD52_GAP_X_TRAIN,
    y,
    splits
)

display(OLD52_GAP_RESULTS.round(4))


  ensemble member 1/3 (weight=0.333)
  ensemble member 2/3 (weight=0.333)
  ensemble member 3/3 (weight=0.333)
  ensemble member 1/3 (weight=0.333)
  ensemble member 2/3 (weight=0.333)
  ensemble member 3/3 (weight=0.333)
  ensemble member 1/3 (weight=0.333)
  ensemble member 2/3 (weight=0.333)
  ensemble member 3/3 (weight=0.333)


,model,P@R0.7_mean,P@R0.7_std,P@R0.7_last,ROC-AUC,PR-AUC
0,OLD52_GAP_Ensemble(LR+RF+HistGB),0.7182,0.0062,0.7264,0.9296,0.7520
1,OLD52_GAP_HistGB,0.6684,0.0250,0.7032,0.9228,0.7431
2,OLD52_GAP_RandomForest,0.6684,0.0182,0.6798,0.9207,0.7332
3,OLD52_GAP_LogReg,0.4969,0.0220,0.4709,0.9066,0.6161


### Сравнение моделей
Сохраняю mean, std и результат последнего фолда.

Обучаю одиночные модели на каждом фолде.

In [44]:
def summarize_fold_predictions(fold_preds, y_folds):
    p_scores, roc_scores, pr_scores = [], [], []
    for p, y_va in zip(fold_preds, y_folds):
        p_scores.append(precision_at_recall(y_va, p))
        roc_scores.append(roc_auc_score(y_va, p))
        pr_scores.append(average_precision_score(y_va, p))
    return {
        'P@R0.7_mean': float(np.mean(p_scores)),
        'P@R0.7_std': float(np.std(p_scores)),
        'P@R0.7_last': float(p_scores[-1]),
        'ROC-AUC': float(np.mean(roc_scores)),
        'PR-AUC': float(np.mean(pr_scores)),
    }


fold_y = [y[va_mask] for _, va_mask in splits]
fold_predictions = {}
single_rows = []

for name, (feature_set, factory) in MODEL_SPECS.items():
    X_all = X_BY_SET[feature_set]
    preds = []
    for fold_id, (tr_mask, va_mask) in enumerate(splits, 1):
        model = factory()
        model.fit(X_all.loc[tr_mask], y[tr_mask])
        preds.append(model.predict_proba(X_all.loc[va_mask])[:, 1])
    fold_predictions[name] = preds
    single_rows.append({'model': name, **summarize_fold_predictions(preds, fold_y)})
    print(
        f'{name:18s} | mean={single_rows[-1]["P@R0.7_mean"]:.4f} '
        f'| std={single_rows[-1]["P@R0.7_std"]:.4f} '
        f'| last={single_rows[-1]["P@R0.7_last"]:.4f}'
    )

single_results = (
    pd.DataFrame(single_rows)
    .sort_values(['P@R0.7_mean', 'P@R0.7_last'], ascending=False)
    .reset_index(drop=True)
)

single_results.round(4)

BASE_LogReg        | mean=0.5239 | std=0.0212 | last=0.4952
BASE_RF            | mean=0.6720 | std=0.0075 | last=0.6826
BASE_ExtraTrees    | mean=0.5233 | std=0.0103 | last=0.5292
BASE_HistGB        | mean=0.6817 | std=0.0048 | last=0.6828
EXT_HistGB         | mean=0.6989 | std=0.0082 | last=0.6982
BASE_LGBM_A        | mean=0.7010 | std=0.0170 | last=0.7243
BASE_LGBM_B        | mean=0.7181 | std=0.0176 | last=0.7311
BASE_CAT_A         | mean=0.7146 | std=0.0073 | last=0.7243
EXT_CAT_A          | mean=0.7392 | std=0.0091 | last=0.7476
EXT_CAT_B          | mean=0.7096 | std=0.0106 | last=0.7196
EXT_CAT_C          | mean=0.7330 | std=0.0185 | last=0.7393
  ensemble member 1/3 (weight=0.429)
  ensemble member 2/3 (weight=0.286)
  ensemble member 3/3 (weight=0.286)
  ensemble member 1/3 (weight=0.429)
  ensemble member 2/3 (weight=0.286)
  ensemble member 3/3 (weight=0.286)
  ensemble member 1/3 (weight=0.429)
  ensemble member 2/3 (weight=0.286)
  ensemble member 3/3 (weight=0.286)
BASE_En

,model,P@R0.7_mean,P@R0.7_std,P@R0.7_last,ROC-AUC,PR-AUC
0,EXT_CAT_A,0.7392,0.0091,0.7476,0.9378,0.7801
1,EXT_LGBM_B,0.7368,0.0080,0.7440,0.9287,0.7718
2,EXT_CAT_C,0.7330,0.0185,0.7393,0.9352,0.7732
3,EXT_LGBM_D,0.7264,0.0101,0.7404,0.9279,0.7672
4,BASE_Ensemble(LGBM+RF+HistGB),0.7261,0.0118,0.7406,0.9265,0.7567
5,EXT_LGBM_C,0.7254,0.0089,0.7130,0.9288,0.7706
6,EXT_LGBM_A,0.7219,0.0059,0.7136,0.9289,0.7722
7,BASE_LGBM_B,0.7181,0.0176,0.7311,0.9239,0.7552
8,BASE_CAT_A,0.7146,0.0073,0.7243,0.9308,0.7667
9,EXT_CAT_B,0.7096,0.0106,0.7196,0.9380,0.7798


результаты одиночных моделей

In [45]:
single_results = (
    pd.DataFrame(single_rows)
    .sort_values(['P@R0.7_mean', 'P@R0.7_last'], ascending=False)
    .reset_index(drop=True)
)

## 6. Ансамбли
Пробую probability и rank blending без нового fit.

Строю probability и rank ансамбли.

In [46]:
def rank_scores(p):
    # Percentile ranks сохраняют направление score и устойчивы к разным масштабам моделей.
    return pd.Series(p).rank(method='average', pct=True).to_numpy()


def blend_fold_predictions(member_names, weights, rank_mode=False):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    blended = []
    for fold_id in range(len(splits)):
        parts = []
        for name, weight in zip(member_names, weights):
            p = fold_predictions[name][fold_id]
            parts.append(rank_scores(p) if rank_mode else p)
        blended.append(sum(w * p for w, p in zip(weights, parts)))
    return blended


blend_predictions = {}
blend_defs = {}


def add_blend(name, members, weights, rank_mode=False):
    if name in fold_predictions or name in blend_predictions:
        return
    preds = blend_fold_predictions(members, weights, rank_mode=rank_mode)
    blend_predictions[name] = preds
    blend_defs[name] = {
        'members': list(members),
        'weights': list(weights),
        'rank_mode': bool(rank_mode),
    }

# Воспроизводим устойчивые конструкции из предыдущих экспериментов.
add_blend(
    'BASE_Ensemble(LR+RF+HistGB)',
    ['BASE_LogReg', 'BASE_RF', 'BASE_HistGB'],
    [1.0, 1.5, 1.5],
    rank_mode=False,
)
add_blend(
    'BASE_Ensemble(LGBM+RF+HistGB)',
    ['BASE_LGBM_A', 'BASE_RF', 'BASE_HistGB'],
    [1.5, 1.0, 1.0],
    rank_mode=False,
)

# Две заранее заданные конструкции с разными типами бустинга.
add_blend(
    'EXT_Ensemble(LGBM+CAT+HistGB)',
    ['EXT_LGBM_A', 'EXT_CAT_A', 'EXT_HistGB'],
    [1.0, 1.0, 1.0],
    rank_mode=False,
)
add_blend(
    'EXT_RankEnsemble(LGBM+CAT+HistGB)',
    ['EXT_LGBM_A', 'EXT_CAT_A', 'EXT_HistGB'],
    [1.0, 1.0, 1.0],
    rank_mode=True,
)

# Пул для автоматического поиска blend: top по mean + top по last.
pool = list(dict.fromkeys(
    single_results.nlargest(5, 'P@R0.7_mean')['model'].tolist()
    + single_results.nlargest(5, 'P@R0.7_last')['model'].tolist()
))

# Pairwise weighted probability/rank blends.
weight_grid = [0.5, 0.75, 1.0, 1.5, 2.0]
for i in range(len(pool)):
    for j in range(i + 1, len(pool)):
        a, b = pool[i], pool[j]
        for wb in weight_grid:
            add_blend(
                f'Blend({a}+{b};w=1:{wb};prob)',
                [a, b], [1.0, wb], rank_mode=False
            )
            add_blend(
                f'Blend({a}+{b};w=1:{wb};rank)',
                [a, b], [1.0, wb], rank_mode=True
            )

# Three-way blends из самых устойчивых одиночных моделей.
top3_pool = single_results.sort_values(
    ['P@R0.7_mean', 'P@R0.7_last'], ascending=False
).head(6)['model'].tolist()
from itertools import combinations
for members in combinations(top3_pool, 3):
    members = list(members)
    add_blend(
        'Blend(' + '+'.join(members) + ';equal;prob)',
        members, [1, 1, 1], rank_mode=False
    )
    add_blend(
        'Blend(' + '+'.join(members) + ';equal;rank)',
        members, [1, 1, 1], rank_mode=True
    )

blend_rows = []
for name, preds in blend_predictions.items():
    metrics = summarize_fold_predictions(preds, fold_y)
    blend_rows.append({'model': name, **metrics})

blend_results = pd.DataFrame(blend_rows)
all_results = pd.concat([single_results, blend_results], ignore_index=True)
all_results['robust_score'] = np.minimum(
    all_results['P@R0.7_mean'], all_results['P@R0.7_last']
)
all_results = all_results.sort_values(
    ['robust_score', 'P@R0.7_mean', 'P@R0.7_std'],
    ascending=[False, False, True]
).reset_index(drop=True)

print(f'Проверено кандидатов: {len(all_results)}')
display(all_results.head(20).round(4))

Проверено кандидатов: 159


,model,P@R0.7_mean,P@R0.7_std,P@R0.7_last,ROC-AUC,PR-AUC,robust_score
0,Blend(EXT_CAT_A+EXT_LGBM_B+BASE_Ensemble(LGBM+...,0.7661,0.0039,0.7685,0.9348,0.7815,0.7661
1,Blend(EXT_CAT_A+EXT_LGBM_B+BASE_Ensemble(LGBM+...,0.7654,0.0015,0.7662,0.9358,0.7815,0.7654
2,Blend(EXT_CAT_A+BASE_Ensemble(LGBM+RF+HistGB);...,0.7647,0.0074,0.7739,0.9371,0.7818,0.7647
3,Blend(EXT_LGBM_B+BASE_Ensemble(LGBM+RF+HistGB)...,0.7636,0.0097,0.7739,0.9305,0.7703,0.7636
4,Blend(EXT_CAT_A+BASE_Ensemble(LGBM+RF+HistGB);...,0.7635,0.0115,0.7739,0.9355,0.7771,0.7635
5,Blend(EXT_LGBM_B+BASE_Ensemble(LGBM+RF+HistGB)...,0.7626,0.0095,0.7739,0.9309,0.7717,0.7626
6,Blend(EXT_CAT_A+BASE_Ensemble(LGBM+RF+HistGB);...,0.7626,0.0115,0.7778,0.9364,0.7806,0.7626
7,Blend(EXT_CAT_A+EXT_LGBM_D+BASE_Ensemble(LGBM+...,0.7618,0.0023,0.7635,0.9353,0.7801,0.7618
8,Blend(EXT_LGBM_D+BASE_Ensemble(LGBM+RF+HistGB)...,0.7617,0.0152,0.7817,0.9305,0.7699,0.7617
9,Blend(EXT_LGBM_B+BASE_Ensemble(LGBM+RF+HistGB)...,0.7613,0.0201,0.7868,0.9308,0.7754,0.7613


## 7. Финальное обучение и submission
Финальные кандидаты обучаются на всём train.

Обучаю итоговых кандидатов и сохраняю submission.

In [47]:
# -------- Рекурсивное обучение кандидата на 100% train --------
full_models = {}
full_test_single_scores = {}

# Сначала собираем все одиночные модели, которые встречаются в top-кандидатах.
defs_needed = set()
for candidate_name in all_results.head(30)['model']:
    if candidate_name in MODEL_SPECS:
        defs_needed.add(candidate_name)
    elif candidate_name in blend_defs:
        defs_needed.update(blend_defs[candidate_name]['members'])

for name in defs_needed:
    feature_set, factory = MODEL_SPECS[name]
    model = factory()
    model.fit(X_BY_SET[feature_set], y)
    full_models[name] = model
    # Test features для этой же feature_set.
    Xtest = X_test_base if feature_set == 'base' else X_test_ext
    full_test_single_scores[name] = model.predict_proba(Xtest)[:, 1]


def build_test_score(candidate_name):
    if candidate_name in full_test_single_scores:
        return full_test_single_scores[candidate_name]
    spec = blend_defs[candidate_name]
    parts = []
    weights = np.asarray(spec['weights'], dtype=float)
    weights /= weights.sum()
    for member, weight in zip(spec['members'], weights):
        p = full_test_single_scores[member]
        parts.append(rank_scores(p) if spec['rank_mode'] else p)
    return sum(w * p for w, p in zip(weights, parts))


best_mean_name = all_results.sort_values('P@R0.7_mean', ascending=False).iloc[0]['model']
best_last_name = all_results.sort_values('P@R0.7_last', ascending=False).iloc[0]['model']
best_robust_name = all_results.iloc[0]['model']

candidate_names = [best_mean_name, best_last_name, best_robust_name]
submissions = {}
for name in dict.fromkeys(candidate_names):
    score = build_test_score(name)
    sub = pd.DataFrame({'cookie_id': df_test['cookie_id'].values, 'score': score})
    assert len(sub) == len(test)
    assert sub['cookie_id'].is_unique
    assert set(sub['cookie_id']) == set(test['cookie_id'])
    assert sub['score'].notna().all()
    assert sub['score'].between(0, 1).all()
    submissions[name] = sub

# Основные 3 файла.
submissions[best_mean_name].to_csv('submission_best_mean_0.csv', index=False)
submissions[best_last_name].to_csv('submission_best_last_0.csv', index=False)
submissions[best_robust_name].to_csv('submission_0.csv', index=False)

# Дополнительно сохраняем top-5 разных кандидатов для ограниченного числа попыток сдачи.
top_candidate_names = all_results.head(5)['model'].tolist()
for rank_id, candidate_name in enumerate(top_candidate_names, 1):
    build_test_score(candidate_name)
    score = build_test_score(candidate_name)
    sub = pd.DataFrame({'cookie_id': df_test['cookie_id'].values, 'score': score})
    sub.to_csv(f'submission_candidate_{rank_id}.csv', index=False)

print('BEST MEAN :', best_mean_name)
print('BEST LAST :', best_last_name)
print('BEST ROBUST:', best_robust_name)
print('\nГотовы: submission.csv, submission_best_mean.csv, submission_best_last.csv')
print('Дополнительно: submission_candidate_1.csv ... submission_candidate_5.csv')

# Повторная проверка формата финального файла.
final_sub = submissions[best_robust_name]
assert list(final_sub.columns) == ['cookie_id', 'score']
assert len(final_sub) == len(test)
assert final_sub['cookie_id'].is_unique
assert set(final_sub['cookie_id']) == set(test['cookie_id'])
assert final_sub['score'].between(0, 1).all()
assert final_sub['score'].notna().all()

final_sub.head()

  ensemble member 1/3 (weight=0.429)
  ensemble member 2/3 (weight=0.286)
  ensemble member 3/3 (weight=0.286)
BEST MEAN : Blend(EXT_CAT_A+EXT_LGBM_B+BASE_Ensemble(LGBM+RF+HistGB);equal;rank)
BEST LAST : Blend(EXT_LGBM_B+BASE_Ensemble(LGBM+RF+HistGB)+EXT_LGBM_C;equal;rank)
BEST ROBUST: Blend(EXT_CAT_A+EXT_LGBM_B+BASE_Ensemble(LGBM+RF+HistGB);equal;rank)

Готовы: submission.csv, submission_best_mean.csv, submission_best_last.csv
Дополнительно: submission_candidate_1.csv ... submission_candidate_5.csv


,cookie_id,score
0,ck_315fb710a0e371e7,0.057581
1,ck_a76ee3b3e3e522fd,0.837849
2,ck_94c9a4d382689e82,0.728865
3,ck_8eaf9509ad9462a0,0.072588
4,ck_9a88a5a989cb5bc6,0.396958
